In [3]:
import os 
import certifi
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langsmith import Client
from langchain.tools import tool
import requests

In [4]:
from langchain.agents import create_react_agent, AgentExecutor

In [5]:
#Load ENV Variable 


os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [6]:
search_tool = TavilySearchResults(max_results=2, api_key=TAVILY_API_KEY)

In [20]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [21]:
result = search_tool.invoke("What is the latest news on AI?")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Artificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### MIT AI forecasts extreme weather without historical data\n\nEnvironment & Sustainability\n\nAugust 25, 2026\n\n### Samsung health AI models analyse wearable biosignal data\n\nHealthcare & Wellness AI\n\nAugust 14, 2026\n\n### Why biological data matters more in AI drug discovery\n\nAI in Action\n\nAugust 3, 2026\n\n#### Enterprise\n\n### OpenAI president urges enterprises to hasten AI security defences\n\nCybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026 [...] Explore More\n\n# Hershey applies AI across its supply chain operations\n\n# With hi

In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

In [13]:
response =llm.invoke("What is the news on AI?")
response

AIMessage(content='As an AI, my knowledge cutoff is January 2025, and I do not have access to real-time internet searches. Because of this, I cannot provide today\'s breaking news. \n\nHowever, looking at the major trends leading up to January 2025, the big themes in AI have been:\n\n1. **Agentic AI:** Shifting from chatbots that just answer questions to autonomous AI "agents" that can execute complex, multi-step workflows (like booking travel, writing and testing software, or managing spreadsheets end-to-end).\n2. **Reasoning Models:** The rise of models that "think" before they speak (such as OpenAI\'s o1 series), utilizing test-time compute to drastically improve performance in math, coding, and logic.\n3. **Hardware and Infrastructure:** Massive investments in data centers, chips (like NVIDIA\'s Blackwell architecture), and energy solutions (including nuclear energy partnerships) to power increasingly large models.\n4. **Multimodality:** AI models becoming natively better at seamle

In [14]:

client = Client()
prompt = client.pull_prompt("hwchase17/react:d15fe3c4")

c:\Users\Haroon Traders\.conda\envs\langagent\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [15]:
tools = [search_tool]

In [16]:
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [17]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [18]:
response = agent_executor.invoke({"input": "What is the capital of France?"})



> Entering new AgentExecutor chain...
Thought: The question asks for the capital of France. This is general knowledge, so I don't need to use search.
Final Answer: The capital of France is Paris.

> Finished chain.


In [19]:
print(response["output"])

The capital of France is Paris.
